## **집계연산**

### **PySpark 설정**

In [1]:
# [+] pyspark 불러오기
from pyspark import SparkConf, SparkContext

In [2]:
# [+] SparkConf, SparkContext 객체 생성하기
conf = SparkConf().setMaster("local[*]").setAppName("ReductionOperations")

In [3]:
sc = SparkContext(conf=conf)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/06 14:52:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/06 14:52:45 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### **reduce( ) 메서드**

In [4]:
# [+] x+y를 수행하는 reduce() 
sc.parallelize([1, 2, 3, 4, 5]).reduce(lambda x, y: x + y)

15

In [6]:
# reduce() 는 각 파티션별로 수행됨
# [+] 파티션 수: 2
rdd = sc.parallelize([1,2,3,4,5], 2)
rdd.collect()

[1, 2, 3, 4, 5]

In [7]:
# [+] glom() : 파티션 별 RDD 값들을 배열에 모으는 변환 메서드
rdd.glom().collect()

[[1, 2], [3, 4, 5]]

In [8]:
# [+] 파티션 2개에 대해 x+y를 수행하는 reduce()
rdd.reduce(lambda x, y: x + y)

15

In [9]:
# [+] reduce() 주의점: 파티션 구성에 따라 결과 값이 달라질 수 있음
# 예: lambda x, y: (x * 2) + y
sc.parallelize([1,2,3,4,5]).reduce(lambda x, y: (x * 2) + y)

57

In [10]:
# 파티션 수: 2
sc.parallelize([1,2,3,4,5], 2).reduce(lambda x,y: (x * 2) + y)

33

### **10000 개 정수 난수의 총합 계산하기**

In [11]:
!pip install numpy

In [12]:
# [+] numpy 라이브러리 불러오기
import numpy as np

In [13]:
# [+] [0, 100] 범위의 정수 난수 10000개 생성
x = np.random.randint(0, 101, 10000)
x[:100]

array([ 79,  98,  51,  87,  28,  41,  23,   7,  59,  79,  49,  49,  94,
        63,  55,  72,  31,   6,  98,  29,  25,  22,  88,  23,  77,   3,
        41,  14,  18,  14,  38,  76,  16,  79,  68,  66,  19,  51,  23,
        35,  95,  55,  90,  72,  46,  58,  49,  23,   3,  76,   3,  40,
        54,   3,  55,   5,  82,  67,  88,  36,  76,  36,  57,  85,  43,
         3,  47,  17,  10, 100,  16,  26,  34,  34,  74,  41,  42,  80,
        17,  96,   1,   4,  11,  69,  51,  46,  64,  91,   2,  11,  63,
        85,  36,  66,  49,  32,  30,  13,  76,  29])

In [14]:
# [+] 생성한 난수 배열을 RDD로 변환(파티션 수: 8개)
rdd = sc.parallelize(x, 8)

In [15]:
# 파티션 별 RDD 값 확인
partitioned_data = rdd.glom().collect()
for i, partition in enumerate(partitioned_data):
    print(f"Partition {i}: {partition}\n")

Partition 0: [np.int64(79), np.int64(98), np.int64(51), np.int64(87), np.int64(28), np.int64(41), np.int64(23), np.int64(7), np.int64(59), np.int64(79), np.int64(49), np.int64(49), np.int64(94), np.int64(63), np.int64(55), np.int64(72), np.int64(31), np.int64(6), np.int64(98), np.int64(29), np.int64(25), np.int64(22), np.int64(88), np.int64(23), np.int64(77), np.int64(3), np.int64(41), np.int64(14), np.int64(18), np.int64(14), np.int64(38), np.int64(76), np.int64(16), np.int64(79), np.int64(68), np.int64(66), np.int64(19), np.int64(51), np.int64(23), np.int64(35), np.int64(95), np.int64(55), np.int64(90), np.int64(72), np.int64(46), np.int64(58), np.int64(49), np.int64(23), np.int64(3), np.int64(76), np.int64(3), np.int64(40), np.int64(54), np.int64(3), np.int64(55), np.int64(5), np.int64(82), np.int64(67), np.int64(88), np.int64(36), np.int64(76), np.int64(36), np.int64(57), np.int64(85), np.int64(43), np.int64(3), np.int64(47), np.int64(17), np.int64(10), np.int64(100), np.int64(16),

In [16]:
# [+] x+y 집계 연산
rdd.reduce(lambda x, y: x + y)

np.int64(500858)

### **fold( ) 메서드**
```reduce()```와 유사하며, 초기 값을 지정하여 집계를 수행

In [18]:
# [+] 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(1, lambda x, y: x + y)

33

In [19]:
# [+] 파티션 수가 2개이며, 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4], 2).fold(1, lambda x, y: x + y)

13

In [20]:
# [+] 초기 값이 0이 주어지고, x*y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(0, lambda x, y: x * y)

0

### **groupBy( ) 메서드**
```groupBy()```: 인자 함수를 기준으로 그룹핑을 수행


In [23]:
# [+] 홀수/짝수로 그룹핑을 수행하는 groupBy() 코드
res = sc.parallelize([1, 1, 2, 3, 5, 8]).groupBy(lambda x: x % 2)

In [29]:
# [+] groupBy() 결과 출력하기
res.collect()

[(0, <pyspark.resultiterable.ResultIterable at 0x7198b1be26f0>),
 (1, <pyspark.resultiterable.ResultIterable at 0x7198b1be2ab0>)]

### **aggregate( ) 메서드**

In [25]:
# reduce(), fold(): 입출력 RDD 형식이 같지 않으면 오류가 발생
sc.parallelize([1, 2, 3, 4])\
    .reduce(lambda x, y: (x + y, x * y)) 
    # single value -> pair 생성 시도할 경우 오류

TypeError: can only concatenate tuple (not "int") to tuple

In [26]:
sc.parallelize([1, 2, 3, 4]).fold((0, 0), lambda x, y: (x[0] + y, x[1] + 1))

TypeError: unsupported operand type(s) for +: 'int' and 'tuple'

In [27]:
"""
    aggregate(): 입출력 형식 다른 RDD 에 대한 reduction을 수행
    - zeroValues: 파티션 초기 값
    - seqOp: 각 파티션에 대한 reduction 함수
    - combop: 파티션 결과를 합치는 reduction 함수
""" 

seqOp = (lambda x, y: (x[0] + y, x[1] + 1))
combOp = (lambda x, y: (x[0] + y[0], x[1] + y[1]))

In [28]:
# aggregate() 수행
sc.parallelize([1, 2, 3, 4], 2).aggregate((0,0), seqOp, combOp)

(10, 4)